# nb_mdf_55_mlv_refresh - refresh the MLV lineage on demand and report data quality

In production you normally **do not** orchestrate MLV refresh from a notebook. Use one of:
* Lakehouse > Materialized lake views > **Manage schedules** (refreshes in dependency order, retries transient failures)
* The pipeline activity **Refresh Materialized Lake View** after the Bronze load (refreshes every MLV in the lakehouse)
* The REST job `RefreshMaterializedLakeViews` (what this notebook calls) when another tool owns the schedule

This notebook exists so the demo pipeline (`pl_mdf_orchestrator_mlv`) can refresh and wait in one step, and
so the DQ outcome lands in `meta.run_log` next to the notebook path.

In [ ]:
wait_for_completion = True
timeout_minutes = 45
full_refresh_views = ""     # optional, comma-separated MLV names to force a FULL rebuild first (e.g. after a correction)
parent_run_id = ""

In [ ]:
%run nb_mdf_utils

In [ ]:
import time
import sempy.fabric as fabric

run_id = new_run_id("mlv")
started = utcnow()
for v in [x.strip() for x in full_refresh_views.split(",") if x.strip()]:
    print("FULL refresh:", v)
    spark.sql(f"REFRESH MATERIALIZED LAKE VIEW {v} FULL")

client = fabric.FabricRestClient()
ws, lh = fabric.get_workspace_id(), fabric.get_lakehouse_id()
resp = client.post(f"v1/workspaces/{ws}/lakehouses/{lh}/jobs/refreshMaterializedLakeViews/instances")
if resp.status_code not in (200, 202):
    raise Exception(f"Refresh request failed: {resp.status_code} {resp.text[:500]}")
location = resp.headers.get("Location", "")
print("Refresh job accepted:", location)

status, detail = "Accepted", {}
if wait_for_completion and location:
    path = location.split("api.fabric.microsoft.com/", 1)[-1]
    time.sleep(int(resp.headers.get("Retry-After", 30)))
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        detail = client.get(path).json()
        status = detail.get("status", "Unknown")
        print(utcnow().strftime("%H:%M:%S"), status)
        if status in ("Completed", "Failed", "Cancelled", "Deduped"):
            break
        time.sleep(30)

## Data-quality outcome of this refresh
`quarantine_mlv.dq_rule_summary` is itself an MLV, so it was refreshed in the same lineage run.

In [ ]:
dq_name = "quarantine_mlv.dq_rule_summary"
if spark.catalog.tableExists(dq_name):
    rules = spark.table(tname("meta", "dq_rule")).select("rule_id", "rule_name", "rule_category", "severity", "rule_owner")
    display(spark.table(dq_name).join(rules, "rule_id", "left").orderBy("outcome", F.desc("rows_affected")))

# row counts per generated view (names come from meta.mlv_definition, written by nb_mdf_50)
views = [r["mlv_name"] for r in spark.table(tname("meta", "mlv_definition"))
         .where("status = 'CREATED' AND layer IN ('silver', 'quarantine', 'gold')").orderBy("layer", "mlv_name").collect()]
display(spark.createDataFrame([(v, spark.table(v).count()) for v in views], "mlv string, row_count long"))

final = {"Completed": "SUCCESS", "Accepted": "SUCCESS", "Deduped": "SUCCESS"}.get(status, "FAILED")
log_run(run_id, parent_run_id, "MLV_LINEAGE", "mlv", final, started,
        message=f"RefreshMaterializedLakeViews job {status}; {json.dumps(detail.get('failureReason') or {})[:1000]}")
if final == "FAILED":
    raise Exception(f"MLV refresh ended with status {status}: {detail.get('failureReason')}")
notebookutils.notebook.exit(json.dumps({"run_id": run_id, "status": status}))